# ISS decoding with Spotiflow detection

This notebook uses **Spotiflow for spot detection** and the standard **Starfish Per-Round Max Channel (PRMC) decoder** for barcode assignment. Registration, filtering, normalization, QC, and output formatting remain in the `ISS_decoding` pipeline.

Start with one representative region. The optional comparison section runs the original Starfish BlobDetector into a different output directory, so the two results cannot overwrite one another.

## Expected inputs and the two SpaceTx paths

`REGIONS_ROOT` must contain region folders named `R1`, `R2`, and so on. The notebook supports either starting point:

1. **Build SpaceTx here:** start from `R#/preprocessing/CycleX/4_retiled/` (or its `CARE/` subfolder) plus the original codebook CSV. Set `BUILD_SPACETX = True`.
2. **Use existing SpaceTx:** set `BUILD_SPACETX = False`. Each selected region must already contain `decoding/1_SpaceTX_format/experiment.json` and `codebook.json`. If that tree is stored away from `REGIONS_ROOT`, set `SPACETX_OUTPUT_ROOT` to its parent directory.

SpaceTx formatting is still required by this Starfish-based workflow. Spotiflow replaces only the spot detector; it reads the reference image generated from the SpaceTx image stack.

## Imports and environment check

In [ ]:
from importlib.metadata import version
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from IPython.display import display
from starfish import Experiment

import ISS_decoding.SpaceTx_format as STX
import ISS_decoding.decoding as DEC
import ISS_decoding.qc_metrics as QC

print(f"ISS_decoding: {version('ISS-decoding')}")
print(f"Spotiflow: {version('spotiflow')}")
print(f"Starfish: {version('starfish')}")
print(f"PyTorch: {torch.__version__}")
print(f"PyTorch wheel CUDA: {torch.version.cuda}")
print(f"GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")

## Configuration

Edit this cell before running the remaining cells. `REGIONS_TO_PROCESS = [1]` is the recommended first test.

The channel names and `DECODING_CHANNELS` order must match the acquisition and the numeric channel codes in the codebook CSV. `PIXEL_TO_UM = 1.0` keeps coordinates in pixels; use the microscope pixel size to report physical coordinates in microns.

In [ ]:
# Parent directory containing R1, R2, ...
REGIONS_ROOT = Path("/path/to/experiment")
REGIONS_TO_PROCESS = [1]

# Path A: build SpaceTx from retiled TIFFs and this headerless codebook CSV.
BUILD_SPACETX = False
CODEBOOK_CSV = Path("/path/to/codebook.csv")
USE_CARE_TILES = True
PIXEL_TO_UM = 1.0
CHANNELS = ["AF750", "Cy5", "Cy3", "AF488", "DAPI", "At425"]
DECODING_CHANNELS = ["AF750", "AF488", "Cy3", "Cy5", "At425"]
NUCLEI_CHANNEL = "DAPI"

# Path B: use existing SpaceTx. Leave as None when it is below REGIONS_ROOT.
# Otherwise, point this to a tree containing R#/decoding/1_SpaceTX_format/.
SPACETX_OUTPUT_ROOT = None  # or Path("/path/to/existing_spacetx_tree")

# Expensive operations are guarded so paths and settings can be checked first.
RUN_DECODING = False
RUN_BLOB_COMPARISON = False

## Optional: create SpaceTx from retiled TIFFs

This is the same formatting route used by the default decoding notebook. It is skipped when `BUILD_SPACETX` is false. Existing `experiment.json` and `codebook.json` files are not overwritten by `make_spacetx_format`.

In [ ]:
if BUILD_SPACETX:
    if not REGIONS_ROOT.exists():
        raise FileNotFoundError(f"REGIONS_ROOT does not exist: {REGIONS_ROOT}")
    if not CODEBOOK_CSV.is_file():
        raise FileNotFoundError(f"CODEBOOK_CSV does not exist: {CODEBOOK_CSV}")

    STX.make_spacetx_format(
        input_dir=REGIONS_ROOT,
        codebook_csv=CODEBOOK_CSV,
        regions_to_process=REGIONS_TO_PROCESS,
        output_dir_prefix=SPACETX_OUTPUT_ROOT,
        pixel_to_um=PIXEL_TO_UM,
        channels=CHANNELS,
        DO_decorators=DECODING_CHANNELS,
        nuclei_channel=NUCLEI_CHANNEL,
        CARE=USE_CARE_TILES,
    )
else:
    print("Skipping SpaceTx generation and using existing formatted data.")

## Validate the SpaceTx experiments

This inexpensive check confirms that each selected SpaceTx experiment can be loaded before a model is downloaded or any images are processed.

In [ ]:
data_root = Path(SPACETX_OUTPUT_ROOT) if SPACETX_OUTPUT_ROOT is not None else REGIONS_ROOT
if not data_root.exists():
    raise FileNotFoundError(f"SpaceTx data root does not exist: {data_root}")

available_regions = sorted(
    int(path.name[1:])
    for path in data_root.glob("R*")
    if path.is_dir() and path.name[1:].isdigit()
)
selected_region_numbers = available_regions if REGIONS_TO_PROCESS is None else REGIONS_TO_PROCESS
if not selected_region_numbers:
    raise RuntimeError("No regions were selected or discovered.")

validated_experiments = {}
for region_number in selected_region_numbers:
    region = f"R{region_number}"
    experiment_json = data_root / region / "decoding" / "1_SpaceTX_format" / "experiment.json"
    if not experiment_json.is_file():
        raise FileNotFoundError(f"Missing SpaceTx experiment: {experiment_json}")

    experiment = Experiment.from_json(str(experiment_json))
    validated_experiments[region] = experiment_json
    print(f"{region}: {len(list(experiment.keys()))} tiles; codebook loaded")

validated_experiments

## Configure Spotiflow and the Starfish decoder

The `hybiss` pretrained model is designed for hybridization-based in situ imaging and is the recommended starting point. With `probability_threshold=None`, Spotiflow uses the threshold stored with the model. Set an explicit value between 0 and 1 only after inspecting the first result.

`min_distance` suppresses nearby duplicate peaks. `n_tiles=None` processes the reference image in one pass; use `(2, 2)` or a larger grid if GPU memory is insufficient. `measurement_type` controls how Starfish measures each detected spot's barcode intensity after detection.

In [ ]:
SPOTIFLOW_KWARGS = {
    "model": "hybiss",
    "probability_threshold": None,
    "min_distance": 2,
    "n_tiles": (2, 2),  # set to None if the full image fits comfortably
    "measurement_type": "mean",
}

PIPELINE_KWARGS = {
    "register": False,
    "register_dapi": False,
    "masking_radius": 7,
    "normalization_method": "MH",
}

## Run Spotiflow detection and Starfish PRMC decoding

Set `RUN_DECODING = True` in the configuration cell after validation. The first run may download the pretrained `hybiss` model. The model is then reused across all selected tiles and regions.

Completed tile Parquet files are reused after an interrupted run. Spotiflow + PRMC results are kept under `2_decoded_spotiflow/`, separately from the original BlobDetector results in `2_decoded/`.

In [ ]:
if RUN_DECODING:
    DEC.process_experiment(
        input_dir=REGIONS_ROOT,
        regions_to_process=REGIONS_TO_PROCESS,
        output_dir_prefix=SPACETX_OUTPUT_ROOT,
        decode_mode="PRMC",
        dense=False,
        spot_detection_mode="spotiflow",
        spotiflow_kwargs=SPOTIFLOW_KWARGS,
        **PIPELINE_KWARGS,
    )
else:
    print("Decoding is disabled. Set RUN_DECODING = True when ready.")

## Load and inspect a decoded region

Parquet is the canonical output and preserves array-valued QC columns. The CSV beside it is a compatibility copy. `spotiflow_probability` is the neural detector confidence, while `quality_mean` and `quality_minimum` describe the decoded barcode trace.

In [ ]:
REGION_TO_INSPECT = f"R{selected_region_numbers[0]}"
decoded_dir = data_root / REGION_TO_INSPECT / "decoding" / "2_decoded_spotiflow"
decoded_file = decoded_dir / f"{REGION_TO_INSPECT}_decoded.parquet"
if not decoded_file.is_file():
    raise FileNotFoundError(
        f"Decoded output not found: {decoded_file}. Run the decoding cell first."
    )

reads = pd.read_parquet(decoded_file)
reads["assigned"] = reads["target"].notna()
print(f"Loaded {len(reads):,} detected spots from {decoded_file}")
display(reads.head())

In [ ]:
required_columns = {"spot_detector", "spotiflow_probability"}
missing_columns = required_columns.difference(reads.columns)
if missing_columns:
    raise RuntimeError(f"Missing Spotiflow output columns: {sorted(missing_columns)}")

summary = pd.Series(
    {
        "detected_spots": len(reads),
        "assigned_spots": int(reads["assigned"].sum()),
        "assignment_fraction": reads["assigned"].mean(),
        "median_spotiflow_probability": reads["spotiflow_probability"].median(),
        "median_barcode_quality": reads["quality_mean"].median(),
    },
    name=REGION_TO_INSPECT,
)
display(summary.to_frame("value"))

per_tile = (
    reads.groupby("tile", dropna=False)
    .agg(
        detected_spots=("assigned", "size"),
        assigned_spots=("assigned", "sum"),
        median_detection_probability=("spotiflow_probability", "median"),
        median_barcode_quality=("quality_mean", "median"),
    )
    .reset_index()
)
per_tile["assignment_fraction"] = (
    per_tile["assigned_spots"] / per_tile["detected_spots"]
)
display(per_tile.head(20))

## Detection-confidence and barcode-quality diagnostics

A low-confidence tail dominated by unassigned reads can justify increasing the Spotiflow probability threshold. If high-confidence spots remain unassigned, inspect registration, normalization, and the codebook before making the detector more stringent.

In [ ]:
if len(reads):
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    sns.histplot(
        data=reads,
        x="spotiflow_probability",
        hue="assigned",
        bins=40,
        stat="density",
        common_norm=False,
        element="step",
        ax=axes[0],
    )
    axes[0].set_title("Spotiflow detection confidence")

    sns.scatterplot(
        data=reads.sample(min(len(reads), 20000), random_state=1),
        x="spotiflow_probability",
        y="quality_mean",
        hue="assigned",
        alpha=0.35,
        linewidth=0,
        ax=axes[1],
    )
    axes[1].set_title("Detection confidence vs barcode quality")
    plt.tight_layout()
else:
    print("This region contains no detected spots.")

In [ ]:
qc_reads = reads.copy()
if len(qc_reads):
    quality_per_cycle = np.vstack(qc_reads["quality_all_bases"].to_numpy())
    for cycle_index in range(quality_per_cycle.shape[1]):
        qc_reads[f"qc_cycle{cycle_index + 1}"] = quality_per_cycle[:, cycle_index]

    QC.quality_per_cycle(
        qc_reads,
        cycles=quality_per_cycle.shape[1],
        format_base_quality=True,
    )

## Gene frequencies and spatial inspection

In [ ]:
assigned_reads = reads.loc[reads["assigned"]].copy()
if len(assigned_reads):
    QC.plot_frequencies(assigned_reads, on="target")
    QC.plot_expression(
        assigned_reads,
        key="target",
        xcolumn="xc",
        ycolumn="yc",
        genes="all",
        size=4,
        background="black",
        title_color="white",
        figuresize=(10, 10),
        save=None,
        fmt="pdf",
    )
else:
    print("No assigned reads are available for plotting.")

## Optional: compare with the original Starfish BlobDetector

Set `RUN_BLOB_COMPARISON = True` to run the same selected regions with the original detector. These settings affect only BlobDetector. The output is written to `2_decoded/`, while the Spotiflow result remains in `2_decoded_spotiflow/`.

In [ ]:
if RUN_BLOB_COMPARISON:
    DEC.process_experiment(
        input_dir=REGIONS_ROOT,
        regions_to_process=REGIONS_TO_PROCESS,
        output_dir_prefix=SPACETX_OUTPUT_ROOT,
        decode_mode="PRMC",
        dense=False,
        spot_detection_mode="starfish",
        int_threshold=0.002,
        sigma_vals=(1, 10, 30),
        **PIPELINE_KWARGS,
    )
else:
    print("BlobDetector comparison is disabled.")

In [ ]:
comparison_rows = []
for detector, subdirectory in (
    ("spotiflow", "2_decoded_spotiflow"),
    ("starfish_blob", "2_decoded"),
):
    result_file = (
        data_root
        / REGION_TO_INSPECT
        / "decoding"
        / subdirectory
        / f"{REGION_TO_INSPECT}_decoded.parquet"
    )
    if result_file.is_file():
        detector_reads = pd.read_parquet(result_file)
        comparison_rows.append(
            {
                "detector": detector,
                "detected_spots": len(detector_reads),
                "assigned_spots": int(detector_reads["target"].notna().sum()),
                "assignment_fraction": detector_reads["target"].notna().mean(),
                "median_barcode_quality": detector_reads["quality_mean"].median(),
            }
        )

comparison = pd.DataFrame(comparison_rows)
if len(comparison):
    display(comparison)
else:
    print("No detector outputs are available for comparison yet.")

## Interpreting the comparison and scaling up

Do not choose a detector from total spot count alone. Compare assignment fraction, barcode quality, spatial distribution, obvious duplicate detections, and known negative/background regions. A useful first tuning sequence is:

1. Keep the pretrained model threshold (`probability_threshold=None`) and inspect one region.
2. Adjust `min_distance` if adjacent duplicate peaks are visible.
3. Adjust `probability_threshold` only if the confidence plots show a clearly uninformative tail.
4. Change `n_tiles` for memory management, not biological filtering.

After validation, expand `REGIONS_TO_PROCESS` or set it to `None` to process all regions with exactly the same settings. To combine Spotiflow detection with PoSTcode decoding instead, use `ISS_PoSTcode_decoding.ipynb` and select `spot_detection_mode='spotiflow'`.